**Navigation** : [Index](README.md) | [<< SW-3 C#](SW-03-GraphOperations-CSharp.ipynb) | [SW-03 Python — source du portage](SW-03-GraphOperations-Python.ipynb) | [SW-04-Python Python >>](SW-04-SPARQL-Python.ipynb)

# SW-03-GraphOperations-Rust

## Opérations sur les graphes RDF en Rust avec la famille oxigraph

> Ce notebook est le **jumeau Rust** du notebook [SW-03-GraphOperations-Python](SW-03-GraphOperations-Python.ipynb) (rdflib), lui-même miroir du [SW-03-GraphOperations-CSharp](SW-03-GraphOperations-CSharp.ipynb) (dotNetRDF). Il manipule des **graphes RDF** au moyen des crates **oxrdf**, **oxttl** et **oxrdfxml** — les briques du projet oxigraph qui implémentent le modèle de données abstrait de **RDF 1.1** (Cyganiak, Wood, Lanthaler (eds.), *RDF 1.1 Concepts and Abstract Syntax*, Recommandation W3C, 25 fevrier 2014) avec les formats Turtle, N-Triples et RDF/XML.

### Duree estimee : 50 minutes

A la fin de ce notebook, vous saurez :
1. **Lire** des fichiers RDF dans différents formats (Turtle, N-Triples, RDF/XML) avec les parsers d'oxttl et d'oxrdfxml
2. **Ecrire** des graphes RDF vers plusieurs formats via les serializers de la même famille
3. **Fusionner** des graphes RDF (sémantique d'ensemble, `insert` dans un `Graph` oxrdf)
4. **Sélectionner** des triplets selon différents critères, avec les sélecteurs indexés du `Graph` et les itérateurs Rust (`filter`, `map`, `max_by_key`)
5. **Manipuler** des listes RDF (`rdf:first` / `rdf:rest`) construites à la main — la famille n'a pas d'équivalent de `Collection`, et c'est l'occasion de voir la structure de près

### Prerequis
- Kernel Rust (evcxr) — la cellule `:dep` de tête charge les crates une seule fois
- Avoir complete [SW-02-RDFBasics-Python](SW-02-RDFBasics-Python.ipynb)

> **Note** : ce carnet est le troisième côté du triplet SW-03 (Python/rdflib, C#/dotNetRDF, Rust/oxigraph). La comparaison des trois API est documentee dans la section 6.

***

## 0. Installation et Imports

**oxigraph** est un projet Rust dedie a RDF (base de donnees, moteur SPARQL, parsers). Ce carnet n'utilise pas son crate principal mais ses **sous-crates pures**, 100 % Rust, qui couvrent exactement les besoins d'un cours sur les graphes :

- **oxrdf** — le modèle de données : `NamedNode`, `BlankNode`, `Literal`, `Triple`, `Term`, `NamedOrBlankNode`, et le conteneur `Graph` (ensemble de triplets en mémoire) ;
- **oxttl** — les syntaxes textuelles ligne a ligne : `TurtleParser` / `NTriplesParser` en lecture, `TurtleSerializer` / `NTriplesSerializer` en ecriture ;
- **oxrdfxml** — la syntaxe XML : `RdfXmlParser` / `RdfXmlSerializer`.

Le crate `oxigraph` complet aurait aussi marche, mais il embarque un backend de stockage compile en C++ (RocksDB via `oxrocksdb-sys`) : plus lourd a installer, sans apport pour ce carnet. Les sous-crates portent la même licence (`MIT OR Apache-2.0`) et les mêmes formats.

Tous les exemples du carnet travaillent **hors ligne**, sur deux fichiers fournis avec la serie (`data/Example.ttl`, `data/animals.ttl`). Aucun endpoint SPARQL, aucune connexion : ce que le carnet montre est le comportement du moteur lui-meme, reproductible sans reseau.

Une difference structurelle avec le jumeau Python : là ou rdflib cache les erreurs derriere des exceptions, la famille oxigraph retourne des **`Result`** là ou l'echec est possible — `NamedNode::new` peut echouer si l'IRI est invalide, chaque triplet parsé arrive enveloppe dans un `Result`. Les operations d'ensemble, elles, sont directes : `Graph::insert` et `Graph::remove` rendent un `bool` (insertion effective ou triplet deja present / absent). Le jumeau Python guardait chaque cellule avec `if RDFLIB_AVAILABLE:` ; le jumeau Rust, lui, montre l'écueil `Result` du lot : en evcxr (pas de `main`), les cellules utilisent `unwrap()` pour afficher, et les fonctions utilitaires propagent proprement avec `?`.

In [2]:
// Toutes les crates du carnet sont chargees ICI, en tete, une seule fois :
// chaque :dep declenche une compilation evcxr, les regrouper evite les recompilations.
// NB : oxrdf est epingle en "0.3" car oxttl 0.2.5 depend exactement de oxrdf 0.3.5.
:dep oxrdf = "0.3"
:dep oxttl = "0.2"
:dep oxrdfxml = "0.2"

// Les imports sont groupes en tete ; certains types ne servent qu'a partir de la
// section 5, d'ou le allow (sinon rustc signale un import inutilise a cette cellule).
#[allow(unused_imports)]
use oxrdf::{NamedNode, BlankNode, Literal, Triple, Term, NamedOrBlankNode, Graph};
#[allow(unused_imports)]
use oxttl::{TurtleParser, NTriplesParser, TurtleSerializer, NTriplesSerializer};
#[allow(unused_imports)]
use oxrdfxml::{RdfXmlParser, RdfXmlSerializer};

// Vocabulaires W3C : la famille oxigraph ne fournit pas de constantes RDF/RDFS/XSD
// pre-declarees (contrairement a rdflib.namespace). On les fabrique avec nn().
fn nn(s: &str) -> NamedNode { NamedNode::new(s).expect("IRI invalide") }

println!("oxrdf 0.3 + oxttl 0.2 + oxrdfxml 0.2 prets.");
println!();
println!("Crates charges :");
println!("  oxrdf    : NamedNode, BlankNode, Literal, Triple, Term, NamedOrBlankNode, Graph");
println!("  oxttl    : TurtleParser, NTriplesParser, TurtleSerializer, NTriplesSerializer");
println!("  oxrdfxml : RdfXmlParser, RdfXmlSerializer");
println!("  (vocabulaires W3C definis a la main : nn(), rdf:type, ...)");

oxrdf 0.3 + oxttl 0.2 + oxrdfxml 0.2 prets.


Crates charges :


  oxrdf    : NamedNode, BlankNode, Literal, Triple, Term, NamedOrBlankNode, Graph


  oxttl    : TurtleParser, NTriplesParser, TurtleSerializer, NTriplesSerializer


  oxrdfxml : RdfXmlParser, RdfXmlSerializer


  (vocabulaires W3C definis a la main : nn(), rdf:type, ...)


### Interpretation : imports

| Crate | Types charges | Rôle | Equivalent rdflib | Equivalent dotNetRDF |
|-------|---------------|------|-------------------|----------------------|
| `oxrdf` | `NamedNode`, `BlankNode`, `Literal`, `Triple`, `Term`, `NamedOrBlankNode` | Types de noeuds et triplets (immutables) | `URIRef`, `Literal`, `BNode` | `IUriNode`, `ILiteralNode`… |
| `oxrdf` | `Graph` | Conteneur de triplets (ensemble, en mémoire) | `Graph` | `VDS.RDF.Graph` |
| `oxttl` | `TurtleParser`, `NTriplesParser` + serializers | Parsing / sérialisation Turtle et N-Triples | `g.parse()` / `g.serialize()` | `TurtleParser` + `CompressingTurtleWriter` |
| `oxrdfxml` | `RdfXmlParser`, `RdfXmlSerializer` | Syntaxe RDF/XML | idem, `format="xml"` | `RdfXmlParser` + `RdfXmlWriter` |
| *(aucun)* | Listes RDF : pas d'équivalent de `Collection` — construites à la main en section 5 | | `rdflib.collection.Collection` | `Extensions.GetListItems` |

> Comme les noeuds rdflib, les noeuds oxrdf sont **independants du graphe** (`NamedNode::new("...")` est autonome) : c'est dotNetRDF qui est l'exception, avec ses noeuds lies au graphe. Autre différence visible dès la première cellule : `NamedNode::new` retourne un `Result` — une IRI mal formée est une erreur de **type**, pas une exception au runtime. Nota : le type de sujet s'appelle `NamedOrBlankNode` en oxrdf 0.3 (l'alias `Subject` existe mais est marqué deprecated).

La sortie affiche `oxrdf 0.3 + oxttl 0.2 + oxrdfxml 0.2 prets.` puis le bloc `Crates charges :` qui récapitule les trois briques.

***

## 1. Lecture de fichiers RDF

L'outil central de lecture est le **parser par syntaxe** : `TurtleParser::new().for_slice(&octets)`, `NTriplesParser::new().for_slice(...)`, `RdfXmlParser::new().for_slice(...)` — chacun retourne un **itérateur de `Result<Triple>`**, soit consommé triple par triple (streaming, sans construire le graphe), soit versé dans un `Graph` par `insert`.

> Chaque format correspond a une Recommandation W3C distincte de la famille **RDF 1.1** : Turtle (eds. Prud'hommeaux, Carothers ; auteurs Beckett, Berners-Lee, Prud'hommeaux, Carothers, 2014), N-Triples (eds. Carothers, Seaborne ; auteur Beckett, 2014 — la forme canonique ligne-par-triplet), RDF/XML (ed. Beckett, 2004).

Une difference notable avec les deux jumeaux : les parsers de la famille oxigraph sont **nativement streaming**. rdflib construit toujours le graphe (et le dit : pas de `CountHandler` équivalent) ; dotNetRDF offre ses Handlers pour compter sans stocker ; ici, l'itérateur brut EST la donnée — compter sans stocker est un `filter(...).count()` sur l'itérateur, sans aucune API spéciale. C'est l'enseignement porté par la cellule 1.3.

In [3]:
// 1.1 Chargement depuis un fichier Turtle
// Methode A : octets lus depuis le fichier, verses dans un Graph
let bytes_a = std::fs::read("data/Example.ttl").expect("fichier Example.ttl illisible");
let mut store_a: Graph = Graph::new();
for res in TurtleParser::new().for_slice(&bytes_a) {
    store_a.insert(&res.unwrap());
}

// Methode B : meme contenu, parse en streaming SANS graphe, compte a la volee
let bytes_b = std::fs::read("data/Example.ttl").expect("relecture Example.ttl");
let count_b = TurtleParser::new().for_slice(&bytes_b)
    .filter(|t| t.is_ok())
    .count();

println!("Fichier  : {} triplets", store_a.len());
println!("Flux     : {} triplets", count_b);
println!("Identique: {}", store_a.len() == count_b);

Fichier  : 4 triplets


Flux     : 4 triplets


Identique: true


### Lecture : les deux chemins de lecture donnent le meme compte

Les deux methodes comptent **4 triplets** et `Identique` vaut `true` : que l'on verse le parsing dans un `Graph` (méthode A) ou que l'on compte sur l'itérateur de parsing (méthode B), c'est le **même parser Turtle** qui travaille et le **même ensemble de triplets** qui en sort.

Ce que la variante itérateur achète n'est donc pas un résultat different, c'est une **consommation differente** : elle rend le carnet capable de lire un RDF qui n'est pas sur le disque (une réponse HTTP, un contenu décompressé en mémoire, un message reçu) **sans jamais matérialiser le graphe**. C'est exactement le « handler sans graphe » que rdflib n'expose pas et que dotNetRDF facture via ses Handlers — en Rust, c'est la forme par défaut du parser.

Le type de parser reste explicite dans les deux cas : sans extension de fichier à consulter, l'auto-détection n'a plus de prise, et deviner depuis le contenu devient fragile dès que le document mélange les syntaxes. Le prix de l'accord `bytes` est de nommer la syntaxe — un type par format.

La sortie affiche `Fichier  : 4 triplets`, `Flux     : 4 triplets` puis `Identique: true` : les deux chemins donnent exactement les 4 triplets d'`Example.ttl`.

In [4]:
// 1.2 Parsing depuis une chaine : syntaxe explicite, deux parsers
let triple_nt = b"<http://example.org/a> <http://example.org/b> <http://example.org/c>.";

let count_nt = NTriplesParser::new().for_slice(triple_nt)
    .filter(|t| t.is_ok()).count();
println!("Format explicite (nt)  : {} triplet", count_nt);

// La meme ligne en Turtle (espace avant le point final)
let triple_ttl = b"<http://example.org/a> <http://example.org/b> <http://example.org/c> .";
let count_ttl = TurtleParser::new().for_slice(triple_ttl)
    .filter(|t| t.is_ok()).count();
println!("Format explicite (ttl) : {} triplet", count_ttl);

Format explicite (nt)  : 1 triplet


Format explicite (ttl) : 1 triplet


### Lecture : une seule sémantique, deux syntaxes

La même ligne — un sujet, un prédicat, un objet — est parsée deux fois, une fois par `NTriplesParser`, une fois par `TurtleParser`, et les deux comptages donnent exactement `1 triplet`. C'est la démonstration la plus courte du principe fondateur de RDF : le triplet est l'unité de sens, la syntaxe n'est qu'un habit. N-Triples est le sous-ensemble minimal de Turtle — une ligne par triplet, URI complètes — et Turtle ajoute préfixes, points-virgules et regroupements ; une fois PARSÉ, tout retombe dans le même ensemble de triplets. Deux conséquences pratiques : un graphe chargé depuis `.nt` et le même chargé depuis `.ttl` sont indiscernables, et choisir un format de sérialisation est un choix de lisibilité et de compaction — jamais de sémantique. Le format se discute à l'écriture ; à la lecture, seul l'ensemble de triplets compte.

(Noter au passage que le point final collé à l'IRI, `<…>.` sans espace, est la forme que prescrit la grammaire N-Triples 1.1 : la production `triple ::= subject predicate object '.'` ne place aucune espace avant le point terminal. Un writer qui en met une reste lisible, mais la forme canonique n'en a pas — ce n'est pas une tolérance du parser, c'est la règle.)

La sortie porte les deux lignes `Format explicite (nt)  : 1 triplet` et `Format explicite (ttl) : 1 triplet` : chaque parsing ne produit bien qu'un seul et même triplet.

In [5]:
// 1.3 Comptage : deux postures, avec ou sans graphe en memoire
// Posture 1 (comme rdflib) : charger dans un Graph puis len()
let bytes = std::fs::read("data/animals.ttl").expect("animals.ttl illisible");
let mut store_animals: Graph = Graph::new();
for res in TurtleParser::new().for_slice(&bytes) {
    store_animals.insert(&res.unwrap());
}
println!("animals.ttl : {} triplets (comptes apres chargement en memoire)", store_animals.len());

// Posture 2 (streaming natif) : compter SANS construire le graphe.
// C'est l'equivalent du CountHandler de dotNetRDF — que rdflib n'expose pas :
// ici, pas d'API dediee, l'iterateur EST le handler.
let streamed = TurtleParser::new().for_slice(&bytes)
    .filter(|t| t.is_ok()).count();
println!("animals.ttl : {} triplets (comptes en streaming, sans graphe)", streamed);
println!("Note : les deux postures coexistent ; le choix depend de la taille du fichier.");

animals.ttl : 51 triplets (comptes apres chargement en memoire)


animals.ttl : 51 triplets (comptes en streaming, sans graphe)


Note : les deux postures coexistent ; le choix depend de la taille du fichier.


### Interpretation : Méthodes de lecture

| Méthode | Avantage | Equivalent rdflib | Equivalent dotNetRDF |
|---------|----------|-------------------|----------------------|
| `Graph::new()` + boucle `insert` | Graphe réutilisable en mémoire | `g.parse("file.ttl")` | `TurtleParser.Load(g, ...)` |
| `TurtleParser::for_slice(bytes)` + `count()` | Sans graphe, mémoire constante | *(n'existe pas haut niveau)* | `CountHandler` |
| Un type de parser par syntaxe | Pas d'heuristique de détection | `format=` obligatoire sans extension | `StringParser.Parse(g, str)` |

**Différence notable** : la colonne « gap » change de côté selon le jumeau. rdflib construit systematiquement le graphe (pas de `CountHandler`) ; dotNetRDF offre ses Handlers ; la famille oxigraph rend le streaming **par défaut** — le parser est un itérateur, le graphe est optionnel. La cellule montre les deux postures donnant le même compte : **51 triplets**.

> Le `Graph` oxrdf est **reutilisable** et les insertions s'accumulent (sémantique d'ensemble, cf. section 3).

La sortie affiche deux fois `animals.ttl : 51 triplets` — une fois après chargement en mémoire (`g.len()`), une fois en streaming sans graphe : les deux postures comptent les mêmes triplets.

***

## 2. Ecriture de graphes RDF

L'outil central de sérialisation est le **serializer par syntaxe** : `NTriplesSerializer::new().for_writer(Vec::new())` (idem `TurtleSerializer`, `RdfXmlSerializer`), qui consomme les triplets un par un (`serialize_triple(&t)`) puis rend le tampon avec `finish()`. Pour écrire directement dans un fichier, sérialiser en `Vec<u8>` puis `std::fs::write` — le crate ne fait pas de choix d'E/S à votre place. Une asymétrie à connaître entre crates sœurs : le `finish()` de `NTriplesSerializer` rend le tampon directement, celui de `TurtleSerializer` et `RdfXmlSerializer` rend un `io::Result` (le document XML/Turtle se termine par une écriture possible en échec).

Serialiser n'est pas copier : c'est **projeter un ensemble abstrait de triplets dans une syntaxe concrete**. Le graphe est la donnee ; le texte n'en est qu'une ecriture. D'ou une propriete qui surprend au debut : un aller-retour parse(serialize(g)) preserve **le graphe**, jamais les **octets**. Ordre des triplets, prefixes declares, choix de labels de blank nodes, indentation : tout cela varie d'un writer a l'autre — rdflib, oxigraph et dotNetRDF ne produisent pas les mêmes octets pour le même graphe — sans que le graphe change. C'est pourquoi une comparaison de documents RDF se fait sur les triplets, pas sur le texte.

Les formats ne different donc pas par ce qu'ils peuvent exprimer, mais par ce qu'ils **coutent a lire**. N-Triples est la forme canonique — un triplet par ligne, sans prefixe ni raccourci — verbeuse a l'oeil, sans ambiguite pour une machine. Turtle porte la meme information avec les raccourcis d'ecriture (`@prefix`, `a`, virgules et points-virgules). RDF/XML est le format historique de la specification, indispensable pour lire des jeux de donnees anciens.

Un point qui piege : la famille n'a **pas d'auto-détection de format** — là ou rdflib devine depuis l'extension ou le contenu, le type de parser/serializer est toujours explicite. Moins de commodité, zéro heuristique surprise.

In [6]:
// 2.1 Comparaison de formats de sortie
// On repart du graphe de Example.ttl (4 triplets) construit en 1.1 : store_a
// Trois helpers de sérialisation, un par syntaxe — même squelette, writers différents.
fn to_ntriples(g: &Graph) -> String {
    let mut ser = NTriplesSerializer::new().for_writer(Vec::new());
    for t in g.iter() { ser.serialize_triple(t).unwrap(); }
    String::from_utf8(ser.finish()).unwrap()   // finish() rend le tampon directement
}
fn to_turtle(g: &Graph) -> String {
    let mut ser = TurtleSerializer::new().for_writer(Vec::new());
    for t in g.iter() { ser.serialize_triple(t).unwrap(); }
    String::from_utf8(ser.finish().unwrap()).unwrap()  // ici finish() rend io::Result
}
fn to_rdfxml(g: &Graph) -> String {
    let mut ser = RdfXmlSerializer::new().for_writer(Vec::new());
    for t in g.iter() { ser.serialize_triple(t).unwrap(); }
    String::from_utf8(ser.finish().unwrap()).unwrap()
}

let nt_out = to_ntriples(&store_a);
let ttl_out = to_turtle(&store_a);

println!("=== N-Triples ===");
println!("{}", nt_out);
println!("=== Turtle ===");
println!("{}", ttl_out);

=== N-Triples ===


<http://www.w3.org/TR/rdf-syntax-grammar> <http://example.org/stuff/1.0/editor> _:fa6da9260a71a5f4d0d846307e6ebbf4 .


<http://www.w3.org/TR/rdf-syntax-grammar> <http://purl.org/dc/elements/1.1/title> "RDF/XML Syntax Specification (Revised)" .


_:fa6da9260a71a5f4d0d846307e6ebbf4 <http://example.org/stuff/1.0/homePage> <http://purl.org/net/dajobe/> .


_:fa6da9260a71a5f4d0d846307e6ebbf4 <http://example.org/stuff/1.0/fullname> "Dave Beckett" .


=== Turtle ===


<http://www.w3.org/TR/rdf-syntax-grammar> <http://example.org/stuff/1.0/editor> _:fa6da9260a71a5f4d0d846307e6ebbf4 ;


	<http://purl.org/dc/elements/1.1/title> "RDF/XML Syntax Specification (Revised)" .


_:fa6da9260a71a5f4d0d846307e6ebbf4 <http://example.org/stuff/1.0/homePage> <http://purl.org/net/dajobe/> ;


	<http://example.org/stuff/1.0/fullname> "Dave Beckett" .


### Interpretation : Comparaison des formats de sortie

**N-Triples** : format verbeux — chaque triplet occupe une ligne complete avec les URIs en entier : un triplet par ligne, aucun compactage. Pas de prefixe, pas de compression.

**Turtle** : format compact — un sujet répété n'est écrit qu'une fois, ses prédicats s'enchaînant avec `;`, et la syntaxe sait déclarer des préfixes (`@prefix`) pour abréger les IRIs. Ce second levier n'agit que si le writer reçoit la table : le writer d'oxttl n'émet aucun `@prefix` de lui-même, il faut la lui fournir par `with_prefix` (section 2.3).

Le ratio mesuré sur ce graphe est de **1,2:1** (440 caractères N-Triples pour 365 de Turtle) : ce que Turtle gagne ici, c'est le regroupement du sujet répété, pas les préfixes. Sur un graphe plus grand, le gain apporté par les préfixes dépend entièrement de la table fournie au writer.

Une nuance de style entre les jumeaux : pour **le même graphe**, le writer Turtle d'oxttl n'imprime pas exactement les octets du writer rdflib — ordre des triplets, préfixes retenus, labels des blank nodes (`_:b1` vs `_:n1b37...`) diffèrent. Les deux documents parsent vers le **même ensemble de triplets** : la comparaison d'un RDF se fait sur les triplets, jamais sur le texte (cf. prose de la section 2).

Dans la sortie, les 4 triplets tiennent chacun sur une ligne N-Triples aux URI entières ; le document Turtle les regroupe par sujet avec `;` et une tabulation, mais garde lui aussi les URI entières — il ne porte **aucun** `@prefix`. Les préfixes n'apparaissent qu'au document suivant (section 2.3), celui où le writer les reçoit par `with_prefix`.

In [7]:
// 2.2 RDF/XML : deux methodes pour obtenir une chaine
// Methode 1 : serializer vers Vec<u8>, puis String
let xml_str = to_rdfxml(&store_a);
println!("Methode 1 (chaine)     : {} car.", xml_str.len());

// Methode 2 : ecrire la chaine dans un fichier, puis relire
let tmp = std::env::temp_dir().join("sw3b_example_rust.rdf");
std::fs::write(&tmp, xml_str.as_bytes()).expect("ecriture temp impossible");
let xml_from_file = std::fs::read_to_string(&tmp).expect("relecture temp impossible");

println!("Methode 2 (fichier)    : {} car.", xml_from_file.len());
println!("Identiques             : {}", xml_str == xml_from_file);
println!();
println!("=== RDF/XML (extrait) ===");
println!("{}...", &xml_str[..xml_str.len().min(400)]);

Methode 1 (chaine)     : 695 car.


Methode 2 (fichier)    : 695 car.


Identiques             : true


=== RDF/XML (extrait) ===


<?xml version="1.0" encoding="UTF-8"?>


<rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:its="http://www.w3.org/2005/11/its">


	<rdf:Description rdf:about="http://www.w3.org/TR/rdf-syntax-grammar">


		<editor xmlns="http://example.org/stuff/1.0/" rdf:nodeID="fa6da9260a71a5f4d0d846307e6ebbf4"/>


		<title xmlns="http://purl.org/dc/elements/1.1/">RDF/XML Syntax Specification (Revise...


### Lecture : ecrire dans une chaine ou dans un fichier, même document

Les deux methodes produisent le **même nombre de caractères** et `Identiques` vaut `true` : la sérialisation RDF/XML rend un `Vec<u8>`, l'écrire dans un fichier puis le relire ne change pas un octet. La seule difference est **où va le resultat**, jamais ce qu'il contient — d'où l'égalité stricte.

L'extrait montre au passage une propriété du **writer**, pas une limite du format : RDF/XML 1.1 sait écrire un noeud blanc en ligne — un `nodeElement` sans `rdf:about`, `rdf:ID` ni `rdf:nodeID` désigne un noeud blanc, et `rdf:parseType="Resource"` imbrique sa description. Ce que le sérialiseur fait ici, comme celui de rdflib, c'est choisir l'autre forme : un **identifiant de document** (`rdf:nodeID`) et une description séparée. La sortie montre donc un choix d'écriture, pas une impossibilité de la syntaxe.

En pratique, on choisit la forme **chaine** (`Vec<u8>` en Rust) quand le document part vers un appel réseau, une réponse HTTP ou une comparaison ; la forme **fichier** quand il doit survivre à la cellule, parce qu'elle laisse le writer terminer son travail avant l'E/S et évite de garder tout le document en mémoire vive de l'interpréteur.

*(Nuance de compte : la longueur exacte du document RDF/XML dépend du writer — celle du jumeau Python faisait 630 caractères, celle du writer oxrdfxml en fait 695. Ce que la conformance compare est l'égalité entre les DEUX méthodes, pas la longueur elle-même.)*

La sortie affiche `Methode 1 (chaine)` et `Methode 2 (fichier)` au même compte, `Identiques: true`, et l'extrait `<?xml version="1.0" encoding="UTF-8"?>` en tête du document RDF/XML.

In [8]:
// 2.3 Configuration avancee des writers
// La famille n'a pas d'options de mise en page (pas d'indent parametrable) ;
// le seul levier est with_prefix : imposer la table de prefixes du writer Turtle.
println!("=== Turtle standard ===");
println!("{}", ttl_out);

// Turtle avec prefixes fournis : le writer les utilise au lieu d'en generer
let ser_conf = TurtleSerializer::new()
    .with_prefix("dc", "http://purl.org/dc/elements/1.1/").unwrap()
    .with_prefix("ex", "http://example.org/stuff/1.0/").unwrap();
let mut ser3 = ser_conf.for_writer(Vec::new());
for t in store_a.iter() { ser3.serialize_triple(t).unwrap(); }
let with_pfx = String::from_utf8(ser3.finish().unwrap()).unwrap();
println!("=== Turtle avec prefixes imposes (dc:, ex:) ===");
println!("{}", with_pfx);

=== Turtle standard ===


<http://www.w3.org/TR/rdf-syntax-grammar> <http://example.org/stuff/1.0/editor> _:fa6da9260a71a5f4d0d846307e6ebbf4 ;


	<http://purl.org/dc/elements/1.1/title> "RDF/XML Syntax Specification (Revised)" .


_:fa6da9260a71a5f4d0d846307e6ebbf4 <http://example.org/stuff/1.0/homePage> <http://purl.org/net/dajobe/> ;


	<http://example.org/stuff/1.0/fullname> "Dave Beckett" .


=== Turtle avec prefixes imposes (dc:, ex:) ===


... (2 outputs left out)

<http://www.w3.org/TR/rdf-syntax-grammar> ex:editor _:fa6da9260a71a5f4d0d846307e6ebbf4 ;


	dc:title "RDF/XML Syntax Specification (Revised)" .


_:fa6da9260a71a5f4d0d846307e6ebbf4 ex:homePage <http://purl.org/net/dajobe/> ;


	ex:fullname "Dave Beckett" .


### Interpretation : Writers et configuration

| Format | Helper du carnet | Equivalent rdflib | Lisibilite | Taille |
|--------|------------------|-------------------|------------|--------|
| N-Triples | `to_ntriples(g)` | `g.serialize(format="nt")` | Faible | Verbeuse |
| Turtle | `to_turtle(g)` | `g.serialize(format="turtle")` | Elevee | Compacte |
| RDF/XML | `to_rdfxml(g)` | `g.serialize(format="xml")` | Moyenne | Moyenne |

**Différence notable** : rdflib offre un argument `indent` (JSON-LD) ; dotNetRDF expose `IPrettyPrintingWriter` / `ICompressingWriter`. La famille oxigraph est la plus ascétique des trois : **aucune option de mise en page** — la syntaxe choisie détermine entièrement le rendu. Le seul levier est `with_prefix` : imposer la table de préfixes du writer Turtle (visible dans la sortie : le second document utilise `dc:` et `ex:` fournis ; le premier n'en porte aucun, ses URI restent entières).

> L'API est uniforme côté lecture (un type de parser par syntaxe) comme côté écriture (un type de serializer par syntaxe) : pas d'enum de format, le type EST le format.

La sortie montre le même graphe deux fois en Turtle : d'abord sans aucune déclaration de préfixe (URI entières), puis avec `dc:` / `ex:` imposés par `with_prefix` — même ensemble de triplets, deux habits d'octets.

***

## 3. Fusion de graphes

oxrdf implemente la **semantique d'ensemble** des graphes RDF : le `Graph` est un ensemble de triplets, et `insert` rend un `bool` — `true` si le triplet est nouveau, `false` s'il etait deja present (auquel cas rien ne change). La fusion de deux graphes se fait donc en versant l'un dans l'autre (`insert` de chaque triplet de B dans A), ou en construisant un troisième graphe depuis l'union des itérateurs. Compter les `false` de `insert` donne directement le nombre de doublons refusés — là ou rdflib demandait une soustraction `len(g1)+len(g2)-len(union)`.

Cette semantique a deux consequences qu'il faut avoir vues avant de fusionner autre chose que des jouets.

La premiere est que l'union est **idempotente** : fusionner un graphe avec lui-meme ne change rien, et refusionner un graphe deja fusionne ne change rien non plus. Pour des triplets dont les trois termes sont des IRI ou des litteraux, un import rejoue deux fois ne produit pas de doublons — c'est la definition meme d'un ensemble, et c'est aussi pourquoi la fusion ne peut pas servir a compter des occurrences : deux assertions identiques n'en font qu'une. L'exception est le **blank node** : il est re-identifie a chaque parse, donc rejouer l'import d'un document qui en contient ajoute des triplets au lieu de les dedupliquer — rejouer l'import de `Example.ttl` fait passer le graphe de 4 à 7 triplets, les 3 triplets portant le noeud blanc etant recrees avec une nouvelle identite.

La seconde est que la fusion est une **union, pas une reconciliation**. RDF n'a pas d'identite au-dela de l'IRI : deux noeuds qui designent la meme chose restent deux noeuds dans le graphe fusionne tant que personne n'a ecrit le `owl:sameAs` qui les relie. De meme, un blank node de chaque source reste distinct — l'union ne cherche pas a reconnaitre que deux noeuds anonymes sont « le meme ». Fusionner, ici, c'est juxtaposer sans interpreter.

D'ou le choix entre les deux formes de la cellule : verser B dans A modifie A en place ; construire un graphe C depuis `chain(A, B)` laisse les sources intactes — l'équivalent de l'opérateur `|` de rdflib, exprimé ici par un itérateur `chain`, l'idiome Rust des flux composables.

In [9]:
// 3.1 Fusion de deux graphes
// Volet A : graphe cible neuf, on verse store_a (4) puis store_animals (51)
let n1 = store_a.len();
let n2 = store_animals.len();
let somme = n1 + n2;

println!("Graphe 1 (Example.ttl) : {} triplets", n1);
println!("Graphe 2 (animals.ttl) : {} triplets", n2);

let mut g_fusion: Graph = Graph::new();
for t in store_a.iter() { g_fusion.insert(t); }
let mut doublons = 0;
for t in store_animals.iter() {
    if !g_fusion.insert(t) { doublons += 1; }   // insert rend false sur doublon
}
println!("Apres fusion (insert)  : {} triplets (somme theorique: {})", g_fusion.len(), somme);
println!("Doublons refuses       : {}", doublons);

// Volet B : variante NON destructive — un graphe neuf depuis l'union des DEUX
// parsers (equivalent de l'operateur | de rdflib, via chain), sources intactes
let bytes1 = std::fs::read("data/Example.ttl").unwrap();
let bytes2 = std::fs::read("data/animals.ttl").unwrap();
let mut g_union: Graph = Graph::new();
let it1 = TurtleParser::new().for_slice(&bytes1);
let it2 = TurtleParser::new().for_slice(&bytes2);
for res in it1.chain(it2) {
    g_union.insert(&res.unwrap());
}
println!();
println!("Union non destructive (chain) : {} triplets (sources intactes: {}, {})",
    g_union.len(), n1, n2);

Graphe 1 (Example.ttl) : 4 triplets


Graphe 2 (animals.ttl) : 51 triplets


Apres fusion (insert)  : 55 triplets (somme theorique: 55)


Doublons refuses       : 0


Union non destructive (chain) : 55 triplets (sources intactes: 4, 51)


### Interpretation : verification manuelle

**Somme théorique** : 4 (Example.ttl) + 51 (animals.ttl) = **55**. Aucun triplet commun (les deux fichiers utilisent des namespaces disjoints : `http://example.org/stuff/1.0/` vs `http://example.org/animals#`), donc **0 doublon refusé**. Le résultat `55` confirme la sémantique d'ensemble.

| Aspect | oxrdf | rdflib | dotNetRDF |
|--------|-------|--------|-----------|
| Union | `insert` de chaque triplet de B dans A | `g1 += g2` | `g1.Merge(g2)` |
| Union non destructive | graphe neuf depuis `chain(A, B)` | `g1 \| g2` | copie explicite + `Merge` |
| Dédoublonnage | Automatique (`insert` rend `false`) | Automatique | Automatique |
| Blank nodes | Isolés par graphe | Isolés par graphe | Renommés pour éviter collision |

- Les triplets identiques ne sont **pas dupliques** (sémantique d'ensemble RDF) — et le compte des `insert` rendus `false` mesure les doublons **sans** soustraction de longueurs
- L'idiome Rust de l'union non destructive est l'itérateur `chain` : le graphe cible consomme les deux sources à la suite sans les modifier

La sortie affiche `Apres fusion (insert)  : 55 triplets (somme theorique: 55)` et `Doublons refuses       : 0` : l'union vaut exactement la somme des deux graphes. La ligne `Union non destructive (chain) : 55 triplets (sources intactes: 4, 51)` montre le troisième graphe construit sans toucher aux deux premiers.

***

## 4. Selection de triplets

Le `Graph` oxrdf offre des **selecteurs indexes** : `triples_for_subject(s)`, `triples_for_predicate(p)`, `triples_for_object(o)`, `objects_for_subject_predicate(s, p)` — chacun rend un itérateur de triplets (ou de termes) restreint par construction. À côté, l'idiome Rust naturel sur l'ensemble complet est `g.iter()` puis `filter` / `map` / `max_by_key`, exactement l'équivalent des comprehensions Python et de LINQ.

Le motif `(s, p, o)` avec jokers reste **la** primitive de requete de RDF : tout le reste — selecteurs dérivés, et SPARQL plus tard — se ramene a un motif dont les composantes sont liees ou libres. Lire les choses ainsi rend la transition vers SPARQL mecanique : `?s ex:age ?o` n'est que le motif `(None, ex:age, None)` avec des variables nommees.

Deux proprietes se paient si on les ignore. D'abord, un itérateur se consomme **une fois** en Rust aussi : `filter(...).count()` épuise le flux ; pour le parcourir deux fois, il faut le matérialiser (`collect::<Vec<_>>()`). Ensuite, le cout depend des composantes **liees** : un selecteur indexé ne parcourt que les triplets qui matchent ; `g.iter()` parcourt tout. Sur les ~50 triplets de ce carnet la difference est invisible ; sur un graphe de production, c'est elle qui decide si la requete prend une seconde ou une minute. Derniere particularite evcxr : les selecteurs rendent des vues empruntees (`TripleRef`, `TermRef`) — pour garder une valeur dans une variable de session, on la materialise avec `.into_owned()`.

In [10]:
// 4.1 Selection par predicat et par sujet (selecteurs indexes)
// utilitaires d'affichage court : ex:rex, "Rex", _:b1
fn short_iri(iri: &str) -> String {
    if let Some(local) = iri.strip_prefix("http://example.org/animals#") {
        format!("ex:{}", local)
    } else if let Some(local) = iri.strip_prefix("http://www.w3.org/1999/02/22-rdf-syntax-ns#") {
        format!("rdf:{}", local)
    } else if let Some(local) = iri.strip_prefix("http://www.w3.org/2000/01/rdf-schema#") {
        format!("rdfs:{}", local)
    } else { format!("<{}>", iri) }
}
fn short_term(t: &Term) -> String {
    match t {
        Term::NamedNode(n) => short_iri(n.as_str()),
        Term::BlankNode(b) => format!("_:{}", b.as_str()),
        Term::Literal(l) => format!("{:?}", l.value()),
    }
}
fn short_subj(s: &NamedOrBlankNode) -> String {
    match s {
        NamedOrBlankNode::NamedNode(n) => short_iri(n.as_str()),
        NamedOrBlankNode::BlankNode(b) => format!("_:{}", b.as_str()),
    }
}

let rdf_type = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#type");
// L'iteration d'un Graph n'a pas d'ordre garanti : on collecte puis on trie
// pour une sortie deterministe (et lisible).
let mut type_lines: Vec<String> = Vec::new();
for tr in store_animals.triples_for_predicate(&rdf_type) {
    let t = tr.into_owned();
    type_lines.push(format!("  {} -> {}", short_subj(&t.subject), short_term(&t.object)));
}
println!("=== Triplets rdf:type ===");
type_lines.sort();
for l in &type_lines { println!("{}", l); }
println!("  ({} triplets rdf:type)", type_lines.len());

let rex = nn("http://example.org/animals#rex");
let mut rex_props: Vec<String> = Vec::new();
for tr in store_animals.triples_for_subject(&rex) {
    let t = tr.into_owned();
    rex_props.push(format!("  {} -> {}", short_iri(t.predicate.as_str()), short_term(&t.object)));
}
println!();
println!("=== Proprietes de ex:rex ===");
rex_props.sort();
for l in &rex_props { println!("{}", l); }

=== Triplets rdf:type ===


  ex:Animal -> rdfs:Class


  ex:Bird -> rdfs:Class


  ex:Cat -> rdfs:Class


  ex:Dog -> rdfs:Class


  ex:Mammal -> rdfs:Class


... (12 outputs left out)

  ex:name -> "Rex"


  ex:sound -> "Woof"


  rdf:type -> ex:Dog


()

### Interpretation : Résultats de la sélection

**Sélection par predicat** `rdf:type` : **14 triplets** trouves - 6 declarations de classes (Animal, Mammal, Bird, Dog, Cat, Parrot), 4 declarations de proprietes (name, age, sound, canFly) et 4 instances (rex/Dog, minou/Cat, coco/Parrot, buddy/Dog). Verification manuelle sur `animals.ttl` : 6 declarations `a rdfs:Class` + 4 `a rdf:Property` + 4 `a ex:<Classe>` = 14. Confirme.

**Sélection par sujet** `ex:rex` : **4 proprietes** - type (Dog), name (Rex), age (5), sound (Woof). Rex est un chien de 5 ans qui fait "Woof".

La sortie Rust reprend exactement les valeurs du jumeau Python (mêmes comptes, mêmes littéraux) — la sélection par motif ne dépend pas du langage, seulement du graphe. L'idiome change : là où rdflib offrait `g.triples((None, RDF.type, None))`, oxrdf expose un **selecteur indexé** (`triples_for_predicate`) ; et comme l'itération d'un `Graph` n'a pas d'ordre garanti, les lignes sont collectées puis **triées** pour une lecture déterministe.

La sortie liste les 14 triplets `rdf:type` par ordre alphabétique (de `ex:Animal -> rdfs:Class` à `ex:sound -> rdf:Property`), puis les 4 propriétés de `ex:rex` (`ex:age`, `ex:name`, `ex:sound`, `rdf:type`) — le littéral typé XSD integer arrive par `Literal::value()`.

In [11]:
// 4.2 Selection combinee et iterateurs Rust (equivalent LINQ / comprehensions)
fn literal_value(t: &Term) -> String {
    match t { Term::Literal(l) => l.value().to_string(), other => short_term(other) }
}

let ex_name = nn("http://example.org/animals#name");
let ex_age = nn("http://example.org/animals#age");

println!("=== Noms des animaux ===");
let mut name_lines: Vec<String> = Vec::new();
for tr in store_animals.triples_for_predicate(&ex_name) {
    let t = tr.into_owned();
    name_lines.push(format!("  {} -> {}", short_subj(&t.subject), literal_value(&t.object)));
}
name_lines.sort();
for l in &name_lines { println!("{}", l); }

// Animaux de plus de 5 ans : filter + map, l'equivalent du .Where(...) LINQ
println!();
println!("=== Animaux > 5 ans (filter) ===");
let mut older: Vec<(String, i64)> = store_animals.triples_for_predicate(&ex_age)
    .filter_map(|tr| {
        let t = tr.into_owned();
        let age: i64 = literal_value(&t.object).parse().ok()?;
        Some((short_subj(&t.subject), age))
    })
    .filter(|(_, age)| *age > 5)
    .collect();
older.sort_by(|a, b| b.1.cmp(&a.1));   // age decroissant, sortie deterministe
for (s, age) in &older {
    println!("  {} : {}", s, age);
}

// Animal le plus age : max_by_key, l'equivalent du .Max(...) LINQ
let all_ages: Vec<(String, i64)> = store_animals.triples_for_predicate(&ex_age)
    .filter_map(|tr| {
        let t = tr.into_owned();
        let age: i64 = literal_value(&t.object).parse().ok()?;
        Some((short_subj(&t.subject), age))
    })
    .collect();
let oldest = all_ages.iter().max_by_key(|(_, a)| *a).cloned().unwrap();
println!();
println!("=== Animal le plus age ===");
println!("  {} : {} ans", oldest.0, oldest.1);

=== Noms des animaux ===


  ex:buddy -> Buddy


  ex:coco -> Coco


  ex:minou -> Minou


  ex:rex -> Rex


=== Animaux > 5 ans (filter) ===


  ex:coco : 12


  ex:buddy : 7


=== Animal le plus age ===


  ex:coco : 12 ans


### Interpretation : Méthodes de sélection

| Idiome Rust | Pattern | Equivalent rdflib | Equivalent dotNetRDF |
|-------------|---------|-------------------|----------------------|
| `triples_for_subject(&s)` | `s ? ?` | `g.triples((s, None, None))` | `GetTriplesWithSubject(s)` |
| `triples_for_predicate(&p)` | `? p ?` | `g.triples((None, p, None))` | `GetTriplesWithPredicate(p)` |
| `objects_for_subject_predicate(&s, &p)` | `s p ?` | `g.triples((s, p, None))` | `GetTriplesWithSubjectPredicate` |
| `filter(...).filter_map(parse)` + `max_by_key` | composition | comprehension + `max(key=)` | LINQ `.Where`/`.Max` |

**Verification manuelle** : la sélection `rdf:type` renvoie **14** triplets (cf. cellule précédente), et le filtre "age > 5" doit retourner **coco (12)** et **buddy (7)** - ce que confirme la sortie. L'animal le plus age est **coco** a **12 ans**.

L'écueil Rust de cette cellule est le **passage type** : le littéral `xsd:integer` arrive comme une chaîne (`Literal::value()` rend `&str`), et le filtre numérique exige `parse::<i64>()` — un `filter_map` avale proprement un littéral non numérique là ou le jumeau Python levait une exception `int(o)`. Les itérateurs sont composables exactement comme les générateurs Python et `IEnumerable<T>` LINQ.

La sortie affiche `ex:coco : 12` et `ex:buddy : 7` pour le filtre `age > 5` (ordre d'âge décroissant), puis `ex:coco : 12 ans` comme animal le plus âgé : mêmes valeurs que le jumeau Python, idiom Rust.

***

## 5. Listes RDF

Les listes RDF (RDF collections) sont des structures chainees encodees avec `rdf:first` et `rdf:rest`, terminant par `rdf:nil`. rdflib fournit `rdflib.collection.Collection` et dotNetRDF ses `AssertList`/`GetListItems` ; **la famille oxigraph ne fournit rien** — et le jumeau Rust assume : construire, lire, modifier et détruire la liste **à la main**, triplet par triplet. C'est plus verbeux, et c'est l'occasion de voir ce que les API des deux autres jumeaux encapsulent.

Il faut voir pourquoi cette structure existe : RDF ne connait que des triplets, il n'a **aucune notion de liste**. Une sequence est donc encodee en objets — chaque element devient un noeud anonyme qui porte une valeur (`rdf:first`) et un pointeur vers le noeud suivant (`rdf:rest`), le dernier pointant sur `rdf:nil` qui marque la fin. Une liste de n elements coute 2n triplets : un cout qu'aucune autre structure du modele ne demande, et qu'il faut avoir en tete avant de mettre une sequence de mille elements dans un graphe.

Ce cout a une consequence sur l'acces : il n'existe pas d'index. Lire le troisieme element demande de traverser trois `rdf:rest`, donc un acces en O(n). C'est la raison pour laquelle les listes RDF sont mal aimees des bases de triplets — une requete qui cherche un element doit derouler la chaine, alors qu'une requete sur des triplets individuels tombe directement sur un index.

La construction manuelle Rust change un parametre pedagogique : chaque operation de la cellule est une **fonction nommée** (`read_list`, `build_list`, `list_append`, `list_remove_by_value`, `clear_list`) — là où `Collection` cachait le chaînage, ici chaque `rdf:rest` réécrit est visible dans le code.

In [12]:
// 5.1 Creer et lire une liste RDF (construction manuelle)
let rdf_first = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#first");
let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
let rdf_nil = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#nil");

// Chaine : b1 -(first 1, rest b2)> b2 -(first 2, rest b3)> b3 -(first 3, rest nil)>
let mut store_l: Graph = Graph::new();
let b1 = BlankNode::default();
let b2 = BlankNode::default();
let b3 = BlankNode::default();
let items: Vec<Literal> = vec![1.into(), 2.into(), 3.into()];
let chain: Vec<BlankNode> = vec![b1.clone(), b2.clone(), b3.clone()];
for (idx, node) in chain.iter().enumerate() {
    store_l.insert(&Triple::new(node.clone(), rdf_first.clone(), items[idx].clone()));
    let rest: Term = if idx + 1 < chain.len() {
        Term::BlankNode(chain[idx + 1].clone())
    } else {
        Term::NamedNode(rdf_nil.clone())
    };
    store_l.insert(&Triple::new(node.clone(), rdf_rest.clone(), rest));
}

// utilitaire : deroule la chaine rdf:rest depuis une tete et rend les valeurs
fn read_list(g: &Graph, head: &NamedOrBlankNode) -> Vec<Term> {
    let rdf_first = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#first");
    let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
    let rdf_nil = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#nil");
    let mut values = Vec::new();
    let mut current = head.clone();
    loop {
        let v = g.objects_for_subject_predicate(&current, &rdf_first).next().map(|o| o.into_owned());
        let r = g.objects_for_subject_predicate(&current, &rdf_rest).next().map(|o| o.into_owned());
        if let Some(v) = v { values.push(v); }
        match r {
            Some(Term::NamedNode(n)) if n == rdf_nil => break,
            Some(Term::BlankNode(b)) => current = NamedOrBlankNode::from(b),
            _ => break,
        }
    }
    values
}

fn fmt_items(vals: &[Term]) -> String {
    let strs: Vec<String> = vals.iter().map(|v| match v {
        Term::Literal(l) => l.value().to_string(),
        other => other.to_string(),
    }).collect();
    format!("[{}]", strs.join(", "))
}

let head_l = NamedOrBlankNode::from(b1.clone());
let vals = read_list(&store_l, &head_l);
println!("Items (manual read) : {}", fmt_items(&vals));
println!("Longueur            : {}", vals.len());

// Noeuds intermediaires = cells de la liste chainee
let first_count = store_l.triples_for_predicate(&rdf_first).count();
let rest_count = store_l.triples_for_predicate(&rdf_rest).count();
println!("Noeuds intermediaires : 3");
println!("rdf:first count    : {}", first_count);
println!("rdf:rest count     : {}", rest_count);
println!("Total liste        : {} triplets", first_count + rest_count);

Items (manual read) : [1, 2, 3]


Longueur            : 3


Noeuds intermediaires : 3


rdf:first count    : 3


rdf:rest count     : 3


Total liste        : 6 triplets


### Interpretation : Structure interne

```
head -> [1 | rest] -> [2 | rest] -> [3 | rest] -> rdf:nil
```

| Élément | Méthode (ici, manuelle) | Chez rdflib |
|---------|--------------------------|-------------|
| Valeurs (`rdf:first`) | `read_list(g, &head)` — parcours `rdf:rest` | `list(Collection(g, head))` |
| Noeuds intermediaires | les `BlankNode` créés explicitement (`b1`, `b2`, `b3`) | cachés par `Collection` |
| Tous les triplets | selecteurs `triples_for_predicate(rdf:first/rdf:rest)` | `g.triples((None, RDF.first, None))` |

**Verification manuelle** : une liste de 3 éléments produit **3** `rdf:first` + **3** `rdf:rest` (le dernier `rdf:rest` pointe vers `rdf:nil`) = **6 triplets**. Confirme par la sortie. Chaque élément coute 2 triplets (`rdf:first` + `rdf:rest`).

La sortie affiche `Items (manual read) : [1, 2, 3]` et `Longueur            : 3`, puis `rdf:first count    : 3` et `rdf:rest count     : 3` (`Total liste        : 6 triplets`) : les 3 éléments sont stockés comme 3 nœuds chaînés, soit 6 triplets — mêmes comptes que le jumeau Python, obtenus sans encapsulation.

In [13]:
// 5.2 Construire, ajouter et supprimer des elements (chirurgie de chaine manuelle)
// build_list : l'equivalent de Collection(g, head, items) — autant de blank nodes
// que de valeurs, chaines par rdf:first / rdf:rest, queue sur rdf:nil
fn build_list(g: &mut Graph, values: Vec<Literal>) -> NamedOrBlankNode {
    let rdf_first = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#first");
    let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
    let rdf_nil = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#nil");
    let nodes: Vec<BlankNode> = values.iter().map(|_| BlankNode::default()).collect();
    for (idx, node) in nodes.iter().enumerate() {
        g.insert(&Triple::new(node.clone(), rdf_first.clone(), values[idx].clone()));
        let rest: Term = if idx + 1 < nodes.len() {
            Term::BlankNode(nodes[idx + 1].clone())
        } else { Term::NamedNode(rdf_nil.clone()) };
        g.insert(&Triple::new(node.clone(), rdf_rest.clone(), rest));
    }
    NamedOrBlankNode::from(nodes[0].clone())
}

// list_nodes : deroule la chaine rdf:rest depuis la tete (les cells de la liste)
fn list_nodes(g: &Graph, head: &NamedOrBlankNode) -> Vec<NamedOrBlankNode> {
    let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
    let rdf_nil = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#nil");
    let mut out = vec![head.clone()];
    let mut current = head.clone();
    loop {
        let r = g.objects_for_subject_predicate(&current, &rdf_rest).next().map(|o| o.into_owned());
        match r {
            Some(Term::NamedNode(n)) if n == rdf_nil => return out,
            Some(Term::BlankNode(b)) => { current = NamedOrBlankNode::from(b); out.push(current.clone()); }
            _ => return out,
        }
    }
}

// append : derouler jusqu'au DERNIER noeud (celui dont rdf:rest -> nil), retirer
// son rdf:rest, le faire pointer vers un noeud neuf qui porte la valeur et termine sur nil
fn list_append(g: &mut Graph, head: &NamedOrBlankNode, value: Literal) {
    let rdf_first = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#first");
    let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
    let rdf_nil = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#nil");
    let mut current = head.clone();
    loop {
        let r = g.objects_for_subject_predicate(&current, &rdf_rest).next().map(|o| o.into_owned());
        match r {
            Some(Term::NamedNode(n)) if n == rdf_nil => break,
            Some(Term::BlankNode(b)) => current = NamedOrBlankNode::from(b),
            _ => break,
        }
    }
    let fresh = BlankNode::default();
    g.remove(&Triple::new(current.clone(), rdf_rest.clone(), rdf_nil.clone()));
    g.insert(&Triple::new(current, rdf_rest.clone(), fresh.clone()));
    g.insert(&Triple::new(fresh.clone(), rdf_first, value));
    g.insert(&Triple::new(fresh, rdf_rest, rdf_nil));
}

// remove par valeur : trouver le noeud, retirer ses deux triplets,
// puis recoudre le rdf:rest du precedent sur le suivant du noeud retire
fn list_remove_by_value(g: &mut Graph, head: &NamedOrBlankNode, value: &Literal) {
    let rdf_first = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#first");
    let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
    let nodes = list_nodes(g, head);
    for (i, node) in nodes.iter().enumerate() {
        let v = g.objects_for_subject_predicate(node, &rdf_first).next().map(|o| o.into_owned());
        if matches!(&v, Some(Term::Literal(l)) if l == value) {
            let next: Term = g.objects_for_subject_predicate(node, &rdf_rest).next()
                .map(|o| o.into_owned()).expect("rdf:rest manquant");
            if let Some(Term::Literal(l)) = &v {
                g.remove(&Triple::new(node.clone(), rdf_first.clone(), l.clone()));
            }
            g.remove(&Triple::new(node.clone(), rdf_rest.clone(), next.clone()));
            if i > 0 {
                let prev = nodes[i - 1].clone();
                if let NamedOrBlankNode::BlankNode(bn) = node {
                    g.remove(&Triple::new(prev.clone(), rdf_rest.clone(), bn.clone()));
                }
                g.insert(&Triple::new(prev, rdf_rest, next));
            }
            return;
        }
    }
}

let mut g_l2: Graph = Graph::new();
let head2 = build_list(&mut g_l2, vec![1.into(), 2.into(), 3.into()]);
println!("Avant           : {}", fmt_items(&read_list(&g_l2, &head2)));
list_append(&mut g_l2, &head2, 4.into());
list_append(&mut g_l2, &head2, 5.into());
println!("Apres append    : {}", fmt_items(&read_list(&g_l2, &head2)));
list_remove_by_value(&mut g_l2, &head2, &3.into());
println!("Apres del '3'   : {}", fmt_items(&read_list(&g_l2, &head2)));

Avant           : [1, 2, 3]


Apres append    : [1, 2, 3, 4, 5]


Apres del '3'   : [1, 2, 4, 5]


### Lecture : append puis suppression par valeur

La sortie montre la liste passer de `[1, 2, 3]` à `[1, 2, 3, 4, 5]` après les deux `append`, puis à `[1, 2, 4, 5]` après la suppression de `3` — mêmes états que le jumeau Python. Mais le **code** raconte autre chose : là où `Collection.append` cachait le chaînage, `list_append` le montre — trouver le dernier noeud (celui dont `rdf:rest` pointe vers `rdf:nil`), retirer son `rdf:rest -> nil`, le faire pointer vers un blank node neuf qui porte la valeur et termine la chaîne. Et la suppression par valeur (`list_remove_by_value`) est une **chirurgie de pointeurs** : retirer les deux triplets du noeud, puis recoudre `precedent.rdf:rest` sur le suivant du noeud retiré.

C'est l'enseignement du portage : l'API `Collection` de rdflib (et `AssertList`/`RemoveFromList` de dotNetRDF) ne fait rien d'autre — elle écrit et réécrit ces mêmes triplets `rdf:first`/`rdf:rest`. Le Rust sans équivalent dédié les rend visibles, au prix d'une verbosité assumée.

In [14]:
// 5.3 Cycle complet : creation, ajout, suppression, destruction
// (build_list, list_append et list_remove_by_value sont definis en 5.2 ; il ne
// manque que la destruction — retirer tous les triplets de la chaine)
fn clear_list(g: &mut Graph, head: &NamedOrBlankNode) {
    // RetractList equivalent : retirer tous les triplets rdf:first/rdf:rest des
    // noeuds de la chaine (on collecte d'abord, puis on retire — deux passes,
    // car on ne mutifie pas un graphe pendant qu'on l'emprunte)
    let rdf_first = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#first");
    let rdf_rest = nn("http://www.w3.org/1999/02/22-rdf-syntax-ns#rest");
    for node in list_nodes(g, head) {
        let to_remove: Vec<Triple> = g.triples_for_subject(&node).map(|t| t.into_owned())
            .filter(|t| t.predicate == rdf_first || t.predicate == rdf_rest)
            .collect();
        for t in to_remove { g.remove(&t); }
    }
}

let mut g_l3: Graph = Graph::new();
let head3 = build_list(&mut g_l3, vec!["Alice".into(), "Bob".into(), "Charlie".into()]);
println!("Initial           : {}", fmt_items(&read_list(&g_l3, &head3)));

list_append(&mut g_l3, &head3, "Diana".into());
println!("Apres ajout Diana : {}", fmt_items(&read_list(&g_l3, &head3)));

list_remove_by_value(&mut g_l3, &head3, &"Bob".into());
println!("Apres del Bob     : {}", fmt_items(&read_list(&g_l3, &head3)));

let avant = g_l3.len();
clear_list(&mut g_l3, &head3);
println!("Apres clear()     : {} | triplets {} -> {}", fmt_items(&read_list(&g_l3, &head3)), avant, g_l3.len());

Initial           : [Alice, Bob, Charlie]


Apres ajout Diana : [Alice, Bob, Charlie, Diana]


Apres del Bob     : [Alice, Charlie, Diana]


Apres clear()     : [] | triplets 6 -> 0


### Interpretation : API des listes

| Opération | Ici (manuelle) | rdflib | dotNetRDF |
|-----------|----------------|--------|-----------|
| **Créer** | `build_list(g, items)` — blank nodes + `rdf:first`/`rdf:rest` | `Collection(g, head, [items])` | `g.AssertList(éléments)` |
| **Lire** | `read_list(g, &head)` | `list(Collection(g, head))` | `g.GetListItems(root)` |
| **Ajouter** | `list_append(g, head, v)` — recoud la queue | `col.append(item)` | `g.AddToList(root, éléments)` |
| **Retirer par valeur** | `list_remove_by_value` — court-circuit du pointeur | `del col[col.index(item)]` | `g.RemoveFromList(root, éléments)` |
| **Supprimer tout** | `clear_list` — retire chaque paire de triplets | `col.clear()` | `g.RetractList(root)` |

**Différence notable** : les trois bibliothèques stockent exactement la même chose. La sortie le vérifie — mêmes états intermédiaires que le jumeau Python (`[Alice, Bob, Charlie]` → `+Diana` → `−Bob` → `[]`, triplets `6 -> 0`). Ce qui change est la **visibilité** : chaque `rdf:rest` réécrit par `list_append` est un triplet qu'on voit partir et revenir, là où `Collection` l'exécutait en silence.

> **Quand utiliser les listes RDF ?** : Collections ordonnées (séquences, étapes). Pour des ensembles non ordonnés, préférez des triplets individuels — la liste RDF est coûteuse (2 triplets par élément) et peu query-friendly.

La sortie affiche `Apres del Bob     : [Alice, Charlie, Diana]` puis `Apres clear()     : [] | triplets 6 -> 0` : la destruction manuelle retire bien les 6 triplets de la liste de 3 éléments.

***

## 6. Comparaison cross-langage dotNetRDF vs rdflib vs famille oxigraph

| Aspect | dotNetRDF (C#) | rdflib (Python) | oxrdf/oxttl (Rust) |
|--------|----------------|-----------------|-----------------|
| **Style API** | OOP verbeux, typage fort, interfaces multiples | Pythonique, concis, tuples natifs | Types immuables + itérateurs, `Result` ciblé |
| **Noeuds** | Lies au graphe (`g.CreateUriNode(...)`) | Independants (`URIRef(...)`) | Independants (`NamedNode::new(...)` → `Result`) |
| **Triple** | Objet `new Triple(s, p, o)` | Tuple Python `(s, p, o)` | `Triple::new(s, p, o)` direct (infallible) |
| **Lecture** | Classes par format + Handlers | API unique `g.parse(format=...)` | Un type de parser par syntaxe (itérateur streaming natif) |
| **Ecriture** | Classes par format + interfaces de config | `g.serialize(format=...)` | Un type de serializer par syntaxe (aucune option de mise en page) |
| **Fusion** | `g1.Merge(g2)` (en place) | `g1 += g2` ou `g1 \| g2` | `insert` (rend `false` sur doublon), ou graphe neuf via `chain` |
| **Sélection** | `GetTriplesWithXxx(...)` (5 méthodes) | `g.triples((s, p, o))` | selecteurs indexés + idiome `filter()` / `max_by_key` |
| **Listes RDF** | `Extensions.GetListItems`, `AssertList`, `RetractList` | `rdflib.collection.Collection` | *(aucune)* — construction manuelle (section 5) |
| **Streaming massif** | Handlers (`CountHandler`…) | Construit le graphe (gap) | Itérateur par défaut, le graphe est optionnel |
| **LINQ / comprehensions** | `IEnumerable<Triple>` + LINQ | Générateurs + comprehensions | `filter` / `map` / `max_by_key` sur itérateurs |

### Différences philosophiques

- **dotNetRDF** privilegie l'**explicite** : un type distinct par parser/writer, interfaces de configuration fines, handlers pour le streaming. Avantage : contrôle précis, typage compile. Inconvenient : verbosite.
- **rdflib** privilegie l'**uniformite** : une seule méthode `parse` / `serialize` avec un argument `format`. Avantage : concision. Inconvenient : pas de streaming natif, options limitées.
- **La famille oxigraph** privilegie le **flux** : parsers et serializers sont des itérateurs, le graphe est un choix et non une obligation, et seules les operations qui peuvent echouer portent un `Result` (le reste est direct : `Triple::new` infallible, `insert`/`remove` rendent un `bool`). Avantage : streaming et contrôle mémoire par défaut. Inconvenient : verbosité des `unwrap()`, aucune API de commodité (les listes RDF se construisent à la main).

> **A retenir** : les trois bibliotheques implementent fidelement les mêmes standards W3C (RDF 1.1, Turtle, N-Triples, RDF/XML). Les fichiers produits par l'un sont consommables par les autres — la section 2 montre que les octets diffèrent entre writers alors que les triplets parsés sont identiques.

***

## 7. Exercices pratiques

Mettez en pratique les concepts appris. Chaque **Exemple guide** est une solution complete commentee ; chaque **Exercice** est un squelette a completer.

Les trois exemples guides couvrent les trois gestes du carnet : charger et explorer un graphe, fusionner puis serialiser, construire une liste RDF. Ils s'executent et affichent leur resultat — lisez la sortie avant de passer a l'exercice, c'est elle qui dit ce que la solution produit.

Les exercices s'executent aussi : leur cellule n'echoue pas faute de reponse, elle se contente d'afficher un message tant que le `TODO` n'est pas traite. Completer un exercice consiste donc a ecrire le code entre les reperes, puis a relancer la cellule et a confronter la sortie a ce que l'enonce demande. Aucune erreur volontaire n'est posee dans les squelettes : le carnet reste executable de bout en bout, exercices non faits compris.

### Exemple guide 1 : Charger et explorer

Chargez `data/animals.ttl` et affichez : le nombre total de triplets, les triplets `rdf:type`, et les proprietes de `ex:rex`.

In [15]:
// Exemple guide 1 : Charger animals.ttl et explorer
// (le store_animals de la section 1.3 est deja charge : on le relit ici)
println!("Total triplets : {}", store_animals.len());

// Selection de tous les triplets dont le predicat est rdf:type
// rdf:type indique la classe d'appartenance d'une ressource (ex : rex est de type Dog)
let mut type_lines: Vec<String> = Vec::new();
for tr in store_animals.triples_for_predicate(&rdf_type) {
    let t = tr.into_owned();
    type_lines.push(format!("  {} -> {}", short_subj(&t.subject), short_term(&t.object)));
}
println!("Triplets rdf.type :");
type_lines.sort();
for l in &type_lines { println!("{}", l); }

// Toutes les proprietes de ex:rex
let rex_e1 = nn("http://example.org/animals#rex");
let mut props: Vec<String> = Vec::new();
for tr in store_animals.triples_for_subject(&rex_e1) {
    let t = tr.into_owned();
    props.push(format!("  {} -> {}", short_iri(t.predicate.as_str()), literal_value(&t.object)));
}
println!("Proprietes de rex :");
props.sort();
for l in &props { println!("{}", l); }

Total triplets : 51


Triplets rdf.type :


  ex:Animal -> rdfs:Class


  ex:Bird -> rdfs:Class


  ex:Cat -> rdfs:Class


  ex:Dog -> rdfs:Class


... (12 outputs left out)

  ex:name -> Rex


  ex:sound -> Woof


  rdf:type -> ex:Dog


()

### Lecture : animals.ttl en chiffres

La sortie affiche `Total triplets : 51`, puis les 14 triplets `rdf:type` listés par ordre alphabétique (l'itération d'un `Graph` n'est pas ordonnée ; on trie pour lire), puis les 4 propriétés de `ex:rex` (`ex:age -> 5`, `ex:name -> Rex`, `ex:sound -> Woof`, `rdf:type -> ex:Dog`) — exactement les valeurs du jumeau Python : le graphe est le même, seule la boucle qui le parcourt change de langage.

### Exercice 4 : Recherche avancee dans un graphe

Chargez `data/animals.ttl` et trouvez :

1. Tous les animaux qui sont des chiens (`rdf:type` -> `ex:Dog`)
2. Le nombre d'animaux qui peuvent voler (predicat `ex:canFly` avec valeur `true`)
3. L'animal le plus age (maximum de la valeur du predicat `ex:age`)

**Indices** : pour (1), le selecteur `triples_for_object(&ex_dog)` donne directement les triplets dont l'objet est `ex:Dog` ; pour (3), reprenez le `filter_map` + `max_by_key` de la cellule 4.2.

In [16]:
// TODO etudiant : Chargez data/animals.ttl et trouvez :
// 1. Tous les animaux qui sont des chiens (rdf:type -> ex:Dog)
// 2. Le nombre d'animaux qui peuvent voler (predicat ex:canFly avec valeur true)
// 3. L'animal le plus age (maximum de ex:age)
//
// Indice : pour (1) le selecteur store_animals.triples_for_object(&nn("...#Dog"))
// rend les triplets dont l'objet est ex:Dog ; pour (3) reprenez le filter_map
// + max_by_key de la cellule 4.2
println!("Exercice a completer");

Exercice a completer


### Exemple guide 2 : Fusionner et serialiser

Chargez `data/Example.ttl` et `data/animals.ttl`, fusionnez-les, et ecrivez le résultat en Turtle compact.

In [17]:
// Exemple guide 2 : Fusion + serialization Turtle
let mut e_g1: Graph = Graph::new();
let b_ex = std::fs::read("data/Example.ttl").unwrap();
for res in TurtleParser::new().for_slice(&b_ex) { e_g1.insert(&res.unwrap()); }
let mut e_g2: Graph = Graph::new();
let b_an = std::fs::read("data/animals.ttl").unwrap();
for res in TurtleParser::new().for_slice(&b_an) { e_g2.insert(&res.unwrap()); }

println!("Avant fusion - g1: {}, g2: {}", e_g1.len(), e_g2.len());
// Union : les triplets identiques ne sont pas dupliques (semantique d'ensemble RDF)
let mut refuses = 0;
for t in e_g2.iter() {
    if !e_g1.insert(t) { refuses += 1; }
}
println!("Apres fusion : {} triplets (refuses: {})", e_g1.len(), refuses);

println!("{}", to_turtle(&e_g1));

Avant fusion - g1: 4, g2: 51


Apres fusion : 55 triplets (refuses: 0)


<http://www.w3.org/TR/rdf-syntax-grammar> <http://example.org/stuff/1.0/editor> _:fcf0f06b2b3fad8d709edb086bb0bce ;


	<http://purl.org/dc/elements/1.1/title> "RDF/XML Syntax Specification (Revised)" .


<http://example.org/animals#age> <http://www.w3.org/2000/01/rdf-schema#domain> <http://example.org/animals#Animal> ;


	<http://www.w3.org/2000/01/rdf-schema#range> <http://www.w3.org/2001/XMLSchema#integer> ;


... (47 outputs left out)

	a <http://www.w3.org/1999/02/22-rdf-syntax-ns#Property> ;


	<http://www.w3.org/2000/01/rdf-schema#label> "nom"@fr .


_:fcf0f06b2b3fad8d709edb086bb0bce <http://example.org/stuff/1.0/homePage> <http://purl.org/net/dajobe/> ;


	<http://example.org/stuff/1.0/fullname> "Dave Beckett" .


### Lecture : la fusion sérialisée

La sortie affiche `Avant fusion - g1: 4, g2: 51` puis `Apres fusion : 55 triplets (refuses: 0)` — mêmes comptes que le jumeau Python, et le compte des `insert` refusés confirme zéro doublon. Le Turtle imprimé ensuite énumère ces 55 triplets ; son **habit d'octets** diffère de celui du writer rdflib (préfixes retenus, ordre des sujets, labels des blank nodes) pour le même ensemble de triplets — la comparaison de documents RDF se fait sur les triplets, pas sur le texte.

### Exercice 5 : Fusion avec données en memoire

Chargez `data/animals.ttl` dans un graphe et la chaîne N-Triples suivante dans un autre avec `NTriplesParser::new().for_slice(...)` :

```
<http://example.org/ns#dave> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://example.org/ns#Person> .
<http://example.org/ns#dave> <http://example.org/ns#name> "Dave" .
<http://example.org/ns#dave> <http://example.org/ns#age> "40" .
```

Fusionnez les deux graphes et serialisez le résultat en N-Triples (pas Turtle).

In [18]:
// TODO etudiant : Chargez data/animals.ttl dans un graphe g1
// Chargez la chaine N-Triples ci-dessus dans un graphe g2 (NTriplesParser + for_slice)
// Fusionnez g2 dans g1 (insert de chaque triplet) et affichez le nombre total de triplets
// Serialisez le resultat en N-Triples (pas Turtle) avec to_ntriples
//
// Indice : reprenez la boucle de la cellule 1.1 et le helper to_ntriples (cellule 2.1)
println!("Exercice a completer");

Exercice a completer


### Exemple guide 3 : Liste RDF

Créez une liste `["Alice", "Bob", "Charlie"]`, ajoutez `"Diana"`, supprimez `"Bob"`, affichez le résultat.

In [19]:
// Exemple guide 3 : Liste RDF (helpers de la section 5)
let mut e_store: Graph = Graph::new();
let e_head = build_list(&mut e_store, vec!["Alice".into(), "Bob".into(), "Charlie".into()]);

println!("Initial          : {}", fmt_items(&read_list(&e_store, &e_head)));

// append insere "Diana" en fin de chaine (recoud le rdf:rest du dernier noeud)
list_append(&mut e_store, &e_head, "Diana".into());
println!("Apres ajout      : {}", fmt_items(&read_list(&e_store, &e_head)));

// suppression par valeur : court-circuit du pointeur du precedent
list_remove_by_value(&mut e_store, &e_head, &"Bob".into());
println!("Apres suppression: {}", fmt_items(&read_list(&e_store, &e_head)));

Initial          : [Alice, Bob, Charlie]


Apres ajout      : [Alice, Bob, Charlie, Diana]


Apres suppression: [Alice, Charlie, Diana]


### Lecture : le cycle de vie d'une liste RDF

La sortie enchaîne `[Alice, Bob, Charlie]`, puis `[Alice, Bob, Charlie, Diana]` après l'ajout, puis `[Alice, Charlie, Diana]` après la suppression de `Bob` — mêmes états que le jumeau Python. Chaque opération est immédiatement relue depuis les triplets `rdf:first`/`rdf:rest` du graphe : la liste affichée reflète toujours l'état réel de la chaîne.

### Exercice 6 : Liste RDF personnalisee

Créez une liste RDF `["Paris", "Lyon", "Marseille"]` avec `build_list`, ajoutez `"Toulouse"` avec `list_append`, supprimez `"Lyon"` avec `list_remove_by_value`. Affichez la liste après chaque opération.

In [20]:
// TODO etudiant : Creez une liste RDF ["Paris", "Lyon", "Marseille"] avec build_list
// Ajoutez "Toulouse" avec list_append(g, &head, "Toulouse".into())
// Supprimez "Lyon" avec list_remove_by_value(g, &head, &"Lyon".into())
// Affichez la liste apres chaque operation avec fmt_items(&read_list(g, &head))
println!("Exercice a completer");

Exercice a completer


## References savantes

- **RDF 1.1 Concepts and Abstract Syntax** - Cyganiak, Wood, Lanthaler (eds.), Recommandation W3C, 25 fevrier 2014. Modèle de données abstrait (triplets sujet-predicat-objet, IRIs, blank nodes, litteraux) dont derivent tous les formats manipules dans ce notebook.
- **RDF 1.1 Turtle** - Prud'hommeaux, Carothers (eds.), Beckett, Berners-Lee, Prud'hommeaux, Carothers (auteurs), Recommandation W3C, 25 fevrier 2014. Format de serialization compact lisible par l'humain, le plus utilise en pratique.
- **RDF 1.1 N-Triples** - Carothers, Seaborne (eds.), Beckett (auteur), Recommandation W3C, 25 fevrier 2014. Forme canonique ligne-par-triplet, base d'echange interoperable.
- **oxigraph** (projet) et ses crates **oxrdf**, **oxttl**, **oxrdfxml** (`oxigraph.rs`, licence `MIT OR Apache-2.0`) - modele de donnees et conteneur `Graph` (oxrdf), parsers/serializers Turtle et N-Triples (oxttl), parser/serializer RDF/XML (oxrdfxml) utilises tout au long de ce notebook.

***

## Resume

| Section | Concepts cles | Outils du carnet | Equivalent rdflib / dotNetRDF |
|---------|---------------|------------------|-------------------------------|
| **1. Lecture** | parse, syntaxes, streaming natif | `TurtleParser::for_slice`, `Graph::insert` | `g.parse()` / `TurtleParser` + `CountHandler` |
| **2. Ecriture** | serialize, multi-formats | `NTriplesSerializer`, `TurtleSerializer`, `RdfXmlSerializer` (`for_writer` + `finish`) | `g.serialize()` / writers par classe |
| **3. Fusion** | Union, deduplication | `insert` (rend `false` sur doublon) / `chain` des itérateurs | `+=`, `\|` / `Merge()` |
| **4. Sélection** | selecteurs indexes, composition d'itérateurs | `triples_for_*`, `objects_for_subject_predicate`, `filter_map`, `max_by_key` | `g.triples((s,p,o))` / LINQ |
| **5. Listes** | Collections ordonnees, chaîne `rdf:first`/`rdf:rest` | construction manuelle (`build_list`, `list_append`, …) | `Collection` / `AssertList` |

### Prochaine étape

Dans **[SW-04-SPARQL-Python](SW-04-SPARQL-Python.ipynb)**, nous apprendrons a interroger les graphes RDF avec le langage SPARQL cote Python (equivalent de SW-4 C#). Le moteur SPARQL du projet oxigraph est la suite naturelle côté Rust.

***

**Navigation** : [Index](README.md) | [<< SW-3 C#](SW-03-GraphOperations-CSharp.ipynb) | [SW-03 Python — source du portage](SW-03-GraphOperations-Python.ipynb) | [SW-04-Python Python >>](SW-04-SPARQL-Python.ipynb)